<a href="https://colab.research.google.com/github/Dotunbey/Branham-rag/blob/main/sermon_extracting_form_web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import shutil

mount_point = '/content/drive'

# Checking if the mount_point exists as a regular directory (not a mount) and remove it if it does.
if os.path.exists(mount_point) and not os.path.ismount(mount_point):
    print(f"Warning: Mount point {mount_point} exists but is not an active mount. Attempting to remove it for a clean mount.")
    try:
        shutil.rmtree(mount_point)
        print(f"Successfully removed existing directory at {mount_point}.")
    except Exception as e:
        print(f"Failed to remove {mount_point}: {e}. Drive mount might still fail due to leftover files.")

drive.mount(mount_point, force_remount=True)

Mounted at /content/drive


In [ ]:
import requests
from bs4 import BeautifulSoup
import os


SAVE_DIR = "/content/drive/MyDrive/branham_pdfs"

BASE_URL = "https://branham.org/en/MessageAudio/ENG"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Fetching page...")
response = requests.get(BASE_URL, timeout=30)
soup = BeautifulSoup(response.text, "html.parser")

pdf_links = []

# Extract all PDF links
for a in soup.find_all("a", href=True):
    href = a["href"]

    if href.lower().endswith(".pdf"):
        if href.startswith("http"):
            pdf_links.append(href)
        else:
            pdf_links.append(BASE_URL + "/" + href)

# Deduplicate
pdf_links = list(set(pdf_links))

print(f"Found {len(pdf_links)} PDFs")
print("Downloading...\n")

for idx, url in enumerate(pdf_links, 1):
    filename = url.split("/")[-1]  # Keep exact filename
    filepath = os.path.join(SAVE_DIR, filename)

    try:
        print(f"[{idx}/{len(pdf_links)}] {filename}")

        with requests.get(url, stream=True, timeout=30) as r:
            r.raise_for_status()
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=4096):
                    if chunk:
                        f.write(chunk)

    except Exception as e:
        print(f"FAILED: {filename} — {e}")

print("\n DONE! All PDFs saved in your Google Drive folder:")
print(SAVE_DIR)


Fetching page...
Found 1221 PDFs
Downloading...

[1/1221] e205fa3e907553a6ab93a91e1e07ac7e5010e5799229a6781382e8fca24e5ee079ceb1545f8fb886522d983ded55fd292d8ec7ad9222b5618d319ca409697078.pdf
[2/1221] 5113b3197feb7c044f35e3349395687f37839ec638a9ab2e0a5bd420bded6a1ce37905ca6629023090c952b115b1613ed43ee918083485753ec55af5057256e0.pdf
[3/1221] 10d8ff3988a6e82882af3b684de67993df80a5bae873282d2b4521f97e08caa4c0d99150db059839d1a2d17a53d2f1bd487874e863d55172ab9063e031f898e5.pdf
[4/1221] 648d216fd2ad983b7074213c49e7b933982e2d0bcee82117407417cd5fc73df81cb47d694f8d6245335726a7708ea27e8d3eaa55419b93b95afde375471144f2.pdf
[5/1221] d5aec330082953d697a179a554a0bf9a4880efe36ddb4cf38fd555af158c83425fcdcbf9979060540e1b54a00cca3e7cd6bbf42f188fed1c0712bd5f66c0a381.pdf
[6/1221] a569fd148028be830f8c6aae46ad6d51e7d860a39842903062da9e591b7407077d790c796db868ab58a80358f300929c80c9e6780e8e230eee4cb6071cc93ac8.pdf
[7/1221] 73eb9dbe20896065864c124d62deb80fa47c84ca2e7b42f6aebd941829e4496542d2fe7e630ddeaa1bda4a9db9